# NLP Text Analysis Pipeline

### Extractive Summarization, Sentiment Analysis and Machine Translation

This project implements a compact Natural Language Processing (NLP) pipeline for Spanish text using a combination of classical text-processing techniques and pretrained Transformer models from Hugging Face.

The workflow covers three tasks:

- **Extractive summarization** using sentence relevance scoring
- **Sentiment analysis** using a multilingual BERT model
- **Spanish-to-English machine translation** using a pretrained MarianMT model

The objective is to demonstrate how different NLP approaches can be combined within a single end-to-end text analysis workflow.


## 1. Setup and Input Text

The analysis is performed on a Spanish text discussing the impact of technology on society, including both its benefits and challenges such as the digital divide, privacy and data security.

The original text is intentionally preserved in Spanish because it is used as input for all three NLP tasks.


In [ ]:
# Optional installation for a fresh environment
# !pip install transformers sentencepiece torch

import re
from collections import Counter

import numpy as np
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

text = """
La tecnología ha revolucionado prácticamente todos los aspectos de nuestras vidas, desde cómo nos
comunicamos hasta cómo realizamos nuestras tareas diarias, cambiando también la manera en que
interactuamos con el mundo que nos rodea. Su impacto es evidente en áreas tan diversas como la
educación, la salud, el transporte, y el comercio, mejorando la eficiencia, la accesibilidad y la
conectividad a niveles nunca antes imaginados. Sin embargo, a pesar de los innumerables beneficios que
nos brinda, la tecnología también trae consigo desafíos complejos y significativos que no podemos
ignorar.

Uno de los retos más destacados es la brecha digital, que exacerba las desigualdades sociales y
económicas entre quienes tienen acceso a las tecnologías y quienes no. Esta disparidad limita las
oportunidades educativas, laborales y sociales de muchas personas en diversas partes del mundo,
dejando atrás a comunidades enteras en el desarrollo tecnológico. Además, la expansión de la tecnología
ha planteado serias preocupaciones en torno a la privacidad y la seguridad de los datos personales, ya
que gran parte de nuestras actividades diarias, desde transacciones bancarias hasta interacciones
sociales, dejan un rastro digital que puede ser explotado.

Es crucial reflexionar detenidamente sobre estos aspectos para mitigar los riesgos asociados al avance
tecnológico y garantizar que su impacto sea positivo para la mayor cantidad de personas posible. Sólo a
través de un análisis consciente y acciones responsables podemos aspirar a construir un futuro más
equilibrado, donde la tecnología sea una herramienta al servicio de toda la humanidad, en lugar de
convertirse en una fuente de desigualdad o amenaza.
""".strip()


## 2. Extractive Summarization

A lightweight extractive summarization strategy is used to identify the most representative sentences in the source text.

Instead of generating new text, the method:

1. Splits the document into sentences.
2. Tokenizes words and removes a small set of common Spanish stopwords.
3. Computes word-frequency scores.
4. Scores each sentence according to the normalized importance of its words.
5. Selects the highest-scoring sentences while preserving their original order.

This approach is deterministic, transparent and does not require an additional generative model.


In [ ]:
# Sentence segmentation
sentences = re.split(r'(?<=[.!?])\s+', text)

# Small Spanish stopword set for this lightweight example
stopwords = {
    "a", "al", "algo", "como", "con", "de", "del", "el", "ella", "en", "es",
    "esta", "este", "ha", "la", "las", "lo", "los", "más", "no", "o", "para",
    "por", "que", "se", "sin", "su", "sus", "un", "una", "y", "ya"
}

words = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", text.lower())
content_words = [word for word in words if word not in stopwords]

frequencies = Counter(content_words)
max_frequency = max(frequencies.values())

normalized_frequency = {
    word: count / max_frequency
    for word, count in frequencies.items()
}

sentence_scores = []
for index, sentence in enumerate(sentences):
    sentence_words = re.findall(
        r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+",
        sentence.lower()
    )
    if sentence_words:
        score = sum(
            normalized_frequency.get(word, 0)
            for word in sentence_words
        ) / len(sentence_words)
        sentence_scores.append((index, score))

# Select the three most relevant sentences
top_indices = sorted(
    index
    for index, _ in sorted(
        sentence_scores,
        key=lambda item: item[1],
        reverse=True
    )[:3]
)

summary = " ".join(sentences[index] for index in top_indices)

print("===== EXTRACTIVE SUMMARY =====")
print(summary)


### 2.1 Methodological Note

The summarization component uses a classical extractive approach rather than an abstractive Transformer model.

This choice keeps the pipeline reproducible and interpretable while avoiding model-specific compatibility issues that were encountered during the original experimentation with Spanish summarization checkpoints.

The trade-off is that extractive summarization can only reuse sentences from the original document and cannot generate a more concise reformulation.


## 3. Sentiment Analysis

Sentiment is estimated using the pretrained multilingual model:

`nlptown/bert-base-multilingual-uncased-sentiment`

The model was originally trained to predict ratings from one to five stars. To adapt its output to a simpler sentiment task:

- **1–2 stars → Negative**
- **3 stars → Neutral**
- **4–5 stars → Positive**

To avoid evaluating only the beginning of a long document, predictions are computed sentence by sentence and aggregated using the mean expected star rating.

This should be interpreted as a demonstration of model reuse rather than a domain-specific sentiment classifier.


In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

sentence_predictions = sentiment_model(sentences)

star_values = []
for prediction in sentence_predictions:
    stars = int(prediction["label"].split()[0])
    star_values.append(stars)

mean_stars = float(np.mean(star_values))

if mean_stars >= 3.5:
    sentiment = "positive"
elif mean_stars <= 2.5:
    sentiment = "negative"
else:
    sentiment = "neutral"

print("===== SENTIMENT ANALYSIS =====")
print(f"Aggregated sentiment: {sentiment}")
print(f"Mean star rating: {mean_stars:.2f} / 5")


### 3.1 Interpretation

The source text deliberately contains mixed perspectives: it describes the benefits of technology while also discussing inequality, privacy and data-security risks.

For that reason, the sentiment output should not be interpreted as a complete representation of the document's meaning. The experiment illustrates an important NLP limitation: a generic sentiment model may compress a nuanced argument into a single polarity label.

A more advanced analysis could use aspect-based sentiment analysis or a model fine-tuned specifically for Spanish argumentative text.


## 4. Spanish-to-English Machine Translation

Machine translation is performed with the pretrained MarianMT model:

`Helsinki-NLP/opus-mt-es-en`

The document is translated paragraph by paragraph rather than as one long sequence. This reduces the risk of truncation and keeps each model input within a manageable token length.

Beam search with four beams is used during generation to improve translation quality.


In [ ]:
translation_model_name = "Helsinki-NLP/opus-mt-es-en"

translation_tokenizer = AutoTokenizer.from_pretrained(
    translation_model_name
)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_model_name
)

paragraphs = text.split("\n\n")
translated_paragraphs = []

for paragraph in paragraphs:
    inputs = translation_tokenizer(
        paragraph,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    translated_tokens = translation_model.generate(
        **inputs,
        max_length=500,
        num_beams=4,
        early_stopping=True
    )

    translated_paragraph = translation_tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )

    translated_paragraphs.append(translated_paragraph)

translation = "\n\n".join(translated_paragraphs)

print("===== ENGLISH TRANSLATION =====")
print(translation)


## 5. Conclusions

This project combines three different Natural Language Processing tasks within a single workflow:

- **Extractive summarization** using transparent frequency-based sentence scoring
- **Sentiment analysis** using a pretrained multilingual BERT model
- **Machine translation** using a pretrained MarianMT Transformer

The experiment illustrates the value of combining classical NLP techniques with modern pretrained language models depending on the requirements of each task.

It also highlights several practical considerations:

- NLP models should be evaluated in the context of the domain for which they were trained.
- Long documents may require segmentation to avoid truncation.
- A single sentiment label can oversimplify texts containing multiple perspectives.
- Pretrained Transformer models make sophisticated NLP capabilities accessible without training models from scratch.

Overall, the notebook provides a compact example of an end-to-end NLP pipeline that integrates text preprocessing, model inference and qualitative interpretation.
